## TODOs: Chunking (Step 4)

- Set up imports: `HybridChunker`, `HuggingFaceTokenizer`, `AutoTokenizer`, `DoclingDocument`
- Configure `HybridChunker`: `tokenizer`, `max_tokens=512`, `merge_peers=True`
- Load the JSON saved in Step 3 back into a `DoclingDocument` object via `DoclingDocument.load_from_json(path)` (no re-parsing of the PDF)
- Run `chunker.chunk(dl_doc=doc)`, iterate over chunks, apply `chunker.contextualize()`
- Inspect chunks: count, token distribution (min/max/median), spot-check samples, check shortest/degenerate chunks
- Verify that bbox/page_no metadata is preserved per chunk (`chunk.meta.doc_items`)
- Define a Pydantic schema for a chunk in `schemas.py` (`chunk_id`, `doc_id`, `text`, `contextualized_text`, `page_no`, `bbox`, `heading_path`, `token_count`)
- Generate stable, reproducible chunk IDs
- Save chunks as JSONL (basis for caching in Step 5 and embedding in Step 6)
- Once everything works in the notebook → move the logic into `ingestion.py`

In [9]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
import json
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.types.doc import DoclingDocument
from pathlib import Path
from transformers import AutoTokenizer
from backend.config import settings

In [2]:
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained("BAAI/bge-m3"),
    max_tokens=512,  # explizit setzen, sonst wird model_max_length verwendet
)
hyb_chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,
)

In [5]:
print(settings.DOCUMENTS_OUT_DIR)

C:\Users\admin\Programming\RAG-Systems\classic-RAG\Classic-RAG\data\output_docs


In [20]:
fp = list(settings.DOCUMENTS_OUT_DIR.glob("*.json"))[0]
    
doc = DoclingDocument.load_from_json(fp)

In [ ]:
chunks = []
for i,chunk in enumerate(hyb_chunker.chunk(dl_doc=doc)):
    total = chunk.model_dump(mode="json")
    total["chunk_id"] = f"{i}"
    total["contextualized_text"] = hyb_chunker.contextualize(chunk)
    chunks.append(total)
    

In [31]:
print(chunks[0])

{'text': 'zur  Festlegung  harmonisierter Vorschriften für künstliche Intelligenz und zur Änderung der Verordnungen (EG) Nr. 300/2008, (EU) Nr. 167/2013, (EU) Nr. 168/2013, (EU) 2018/858, (EU) 2018/1139 und (EU) 2019/2144 sowie der Richtlinien 2014/90/EU, (EU) 2016/797 und (EU) 2020/1828 (Verordnung über künstliche Intelligenz)\n(Text von Bedeutung für den EWR)\nDAS EUROPÄISCHE PARLAMENT UND DER RAT DER EUROPÄISCHEN UNION -\ngestützt  auf  den  Vertrag  über  die  Arbeitsweise  der  Europäischen  Union,  insbesondere  auf  die  Artikel  16  und  114,\nauf  Vorschlag  der  Europäischen  Kommission,\nnach Zuleitung  des  Entwurfs  des  Gesetzgebungsakts  an  die  nationalen  Parlamente,\nnach Stellungnahme des Europäischen Wirtschafts- und Sozialausschusses ( 1 ),\nnach Stellungnahme der Europäischen Zentralbank ( 2 ),\nnach Stellungnahme des Ausschusses der Regionen ( 3 ),\ngemäß dem ordentlichen Gesetzgebungsverfahren ( 4 ),\nin  Erwägung  nachstehender  Gründe:', 'meta': {'schema_name